In [1]:
from pathlib import Path

In [ ]:
def rosetta_setup(verbosity = False, allow_overwrite = True, H_optimization = True, extra_params:list[Path] = []):
    flags = [
        f"-mute false" if verbosity else "-mute all",
        f"-ex1",
        f"-ex2",
        f"-no_optH {str(not H_optimization)}",   
        f"-flip_HNQ true",
        f"-ignore_ligand_chi true",
        f"-overwrite" if allow_overwrite else "",
        f"-restore_pre_talaris_2013_behavior true"
    ]
    flags = " ".join(flags)

    extra_flag = []
    for path in extra_params:
        if not path.exists():
            print(f"WARNING: ignoring file {str(path)} (does not exists or is not acessible)")

        else:
            extra_flag.append(str(path))
    if len(extra_flag) > 0:
        flags += " -extra_res_fa " + " ".join(extra_flag)

    try:
        from pyrosetta import init

    except ImportError:
        from pyrosetta_instaler import install_pyrosetta
        from pyrosetta import init

    finally:
        init(flags)
        
def load_rosetta_pose(path:Path):
    from pyrosetta import pose_from_file
    if path.exists():
        return pose_from_file(str(path))

    else:
        raise FileNotFoundError(f"Le fichier {str(path)} n'existe pas ou n'est pas accessible.")

In [ ]:
rosetta_setup(extra_params=[
    Path(".rosettafiles/GSH.params")
], verbosity=False, H_optimization=False)

┌──────────────────────────────────────────────────────────────────────────────┐
│                                 PyRosetta-4                                  │
│              Created in JHU by Sergey Lyskov and PyRosetta Team              │
│              (C) Copyright Rosetta Commons Member Institutions               │
│                                                                              │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRE PURCHASE OF A LICENSE │
│         See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└──────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2024 [Rosetta PyRosetta4.Release.python310.ubuntu 2024.38+release.200d5f9a7d8cdd7afdd078f156da6b5a7d97543f 2024-09-11T17:31:36] retrieved from: http://www.pyrosetta.org


In [5]:
from pyrosetta import get_score_function
scorefxn = get_score_function()

pose_bound = load_rosetta_pose(Path("PDB/GSTD1+GSH.pdb"))
pose_unbound = load_rosetta_pose(Path("PDB/test_GSTD1+GSH_unbound.pdb"))

G_bound = scorefxn(pose_bound)
G_unbound = scorefxn(pose_unbound)
deltaG = G_bound - G_unbound
deltaG

-4.541070813221722

In [11]:
from pyrosetta import Pose, ScoreFunction, Vector1
from pyrosetta import get_fa_scorefxn
from pyrosetta.rosetta import protocols

def binding_affinity(pose:Pose, partners = "AB_C", scorefxn:ScoreFunction = None) -> float:
    """
    Separates two partners and returns the difference of energy between bounded and separated states.
    """
    if scorefxn is None:
        scorefxn = get_fa_scorefxn()

    bind_score = scorefxn(pose)

    # Split partners:
    split_pose = pose.clone()
    jump = 2
    step_size = 100

    protocols.docking.setup_foldtree(pose, partners, Vector1([-1,-1,-1]))
    trans_mover = protocols.rigid.RigidBodyTransMover(split_pose,jump)
    trans_mover.step_size(step_size)
    trans_mover.apply(split_pose)

    split_score = scorefxn(split_pose)

    return bind_score - split_score

In [13]:
binding_affinity(pose_bound)

-4.541070813221722